##Problem 1 - GCNConv

In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.transforms import NormalizeFeatures
import torch.nn as nn
import torch.optim as optim

In [2]:
dataset = Planetoid(root='data/Planetoid', name='Cora', transform=NormalizeFeatures())
data = dataset[0]

In [3]:
print(data)

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


In [4]:
class GNN(nn.Module):
  def __init__(self, in_channels, hidden_channels, out_channels):
    super().__init__()

    self.conv1 = GCNConv(in_channels,hidden_channels)
    self.conv2 = GCNConv(hidden_channels,out_channels)

  def forward(self, x, edge_index):
    x = self.conv1(x,edge_index)
    x = F.relu(x)
    x = F.dropout(x, training=self.training)
    x = self.conv2(x,edge_index)
    return x

In [5]:
print(dataset.num_classes)

7


In [6]:
model = GNN(dataset.num_features, 16, dataset.num_classes)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [7]:
def accuracy(mask):
  model.eval()

  with torch.no_grad():
    output = model(data.x, data.edge_index)
    pred = output.argmax(dim=1)
    correct = (pred[mask] == data.y[mask]).sum()
    acc = int(correct)/int(mask.sum()) * 100

  return acc

In [8]:
epochs = 100

for epoch in range(epochs):
  output = model(data.x, data.edge_index)
  loss = F.cross_entropy(output[data.train_mask], data.y[data.train_mask])
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  train_acc = accuracy(data.train_mask)
  test_acc = accuracy(data.test_mask)

  print(f"Epoch:{epoch+1}, Loss:{loss.item():.4f}, Training Accuracy:{train_acc:.2f}%, Test Accuracy:{test_acc:.2f}%")

Epoch:1, Loss:1.9467, Training Accuracy:19.29%, Test Accuracy:11.50%
Epoch:2, Loss:1.9458, Training Accuracy:27.86%, Test Accuracy:18.50%
Epoch:3, Loss:1.9450, Training Accuracy:30.71%, Test Accuracy:21.70%
Epoch:4, Loss:1.9443, Training Accuracy:35.00%, Test Accuracy:26.80%
Epoch:5, Loss:1.9435, Training Accuracy:42.14%, Test Accuracy:31.10%
Epoch:6, Loss:1.9428, Training Accuracy:50.71%, Test Accuracy:34.70%
Epoch:7, Loss:1.9420, Training Accuracy:57.14%, Test Accuracy:36.90%
Epoch:8, Loss:1.9411, Training Accuracy:62.14%, Test Accuracy:39.50%
Epoch:9, Loss:1.9402, Training Accuracy:65.00%, Test Accuracy:41.30%
Epoch:10, Loss:1.9392, Training Accuracy:67.86%, Test Accuracy:43.40%
Epoch:11, Loss:1.9382, Training Accuracy:70.71%, Test Accuracy:45.50%
Epoch:12, Loss:1.9371, Training Accuracy:74.29%, Test Accuracy:45.90%
Epoch:13, Loss:1.9359, Training Accuracy:75.00%, Test Accuracy:47.30%
Epoch:14, Loss:1.9347, Training Accuracy:75.71%, Test Accuracy:47.10%
Epoch:15, Loss:1.9334, Traini

##Problem 2 - GCNConv + positive edges + negative edges

In [9]:
from torch_geometric.transforms import RandomLinkSplit

In [10]:
split = RandomLinkSplit(num_val=0.0, num_test=0.2, is_undirected=True, add_negative_train_samples=True)
train_data, _, test_data = split(data)

In [11]:
class GCNEncoder(nn.Module):
  def __init__(self, in_channels, hidden_channels):
    super().__init__()

    self.conv1 = GCNConv(in_channels,hidden_channels)
    self.conv2 = GCNConv(hidden_channels,hidden_channels)

  def forward(self, x, edge_index):
    x = self.conv1(x,edge_index)
    x = F.relu(x)
    x = self.conv2(x,edge_index)
    return x

In [12]:
def decode(z, edge_label_index):
  return(z[edge_label_index[0]] * z[edge_label_index[1]]).sum(dim=1)

In [13]:
model = GCNEncoder(dataset.num_features, 16)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [16]:
def test(data):
  model.eval()

  with torch.no_grad():
    output = model(data.x, data.edge_index)
    pred = decode(output, data.edge_label_index)
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    correct = (pred == data.edge_label).sum()
    acc = int(correct)/int(data.edge_label.size(0)) * 100

  return acc

In [18]:
epochs = 100

for epoch in range(epochs):
  output = model(train_data.x, train_data.edge_index)
  pred = decode(output, train_data.edge_label_index)
  loss = F.binary_cross_entropy_with_logits(pred, train_data.edge_label)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  train_acc = test(train_data)
  test_acc = test(test_data)

  print(f"Epoch:{epoch+1}, Loss:{loss.item():.4f}, Training Accuracy:{train_acc:.2f}%, Test Accuracy:{test_acc:.2f}%")

Epoch:1, Loss:0.5109, Training Accuracy:74.05%, Test Accuracy:68.58%
Epoch:2, Loss:0.5099, Training Accuracy:74.08%, Test Accuracy:68.67%
Epoch:3, Loss:0.5089, Training Accuracy:74.18%, Test Accuracy:68.63%
Epoch:4, Loss:0.5079, Training Accuracy:74.22%, Test Accuracy:68.67%
Epoch:5, Loss:0.5069, Training Accuracy:74.32%, Test Accuracy:68.77%
Epoch:6, Loss:0.5060, Training Accuracy:74.39%, Test Accuracy:69.00%
Epoch:7, Loss:0.5051, Training Accuracy:74.46%, Test Accuracy:68.96%
Epoch:8, Loss:0.5041, Training Accuracy:74.60%, Test Accuracy:69.05%
Epoch:9, Loss:0.5032, Training Accuracy:74.66%, Test Accuracy:69.15%
Epoch:10, Loss:0.5024, Training Accuracy:74.69%, Test Accuracy:69.24%
Epoch:11, Loss:0.5015, Training Accuracy:74.70%, Test Accuracy:69.34%
Epoch:12, Loss:0.5007, Training Accuracy:74.73%, Test Accuracy:69.34%
Epoch:13, Loss:0.4998, Training Accuracy:74.78%, Test Accuracy:69.34%
Epoch:14, Loss:0.4990, Training Accuracy:74.89%, Test Accuracy:69.43%
Epoch:15, Loss:0.4982, Traini